# HRC Studio Stem — Kaggle quickstart

Split any track into **lossless 24-bit WAV stems** on a free Kaggle GPU.
This notebook is copy-paste: run the cells top to bottom.

**Before you start**
1. Right sidebar → *Settings* → **Accelerator = GPU T4 x2** (or P100)
2. Same panel → **Internet = On** (needed for the one-time model download)
3. Get the code in, any one of:
   - upload this repo as a Kaggle *Dataset* and attach it (lands in `/kaggle/input/...`), or
   - make the GitHub repo public and let cell 1 clone it, or
   - upload the folder as a zip to `/kaggle/working` and unzip it

The engine is the same code as the local app — no Mac-only dependencies.


## 1 · Locate the repository

In [ ]:
import os, sys, pathlib, subprocess

REPO_URL = "https://github.com/vollstek-ops/hrc-studio-stem.git"   # change if you forked it

def looks_like_repo(p: pathlib.Path) -> bool:
    return (p / "app.py").exists() and (p / "stemkit" / "separator.py").exists()

repo = None
for start in (pathlib.Path.cwd(), pathlib.Path("/kaggle/working")):
    if looks_like_repo(start):
        repo = start
        break
if repo is None:
    for root in ("/kaggle/input", "/kaggle/working"):
        for hit in pathlib.Path(root).rglob("stemkit/separator.py"):
            repo = hit.parent.parent
            break
        if repo:
            break
if repo is None:
    dest = pathlib.Path("/kaggle/working/hrc-studio-stem")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
    repo = dest

os.chdir(repo)
sys.path.insert(0, str(repo))
print("repo:", repo)
print(sorted(p.name for p in repo.iterdir() if not p.name.startswith(".")))


## 2 · Install dependencies

Kaggle images already ship a **CUDA build of PyTorch** — the point of this cell is to add the
audio stack *without* letting pip swap torch for a CPU-only wheel. If `torch.cuda` stops being
available after the install, rerun the cells on a fresh session with `--no-deps` for torch.


In [ ]:
%%bash
set -e
pip install -q --upgrade-strategy only-if-needed \
    "audio-separator>=0.48.0,<0.50" \
    "soundfile>=0.12.1" "numpy>=1.26" \
    "fastapi>=0.110,<1.0" "uvicorn[standard]>=0.27,<1.0" \
    "python-multipart>=0.0.9" "yt-dlp==2026.10.21"
ffmpeg -version | head -1


## 3 · Verify GPU + device selection

In [ ]:
import torch
from stemkit import separator

print("torch:", torch.__version__, "| CUDA build:", torch.version.cuda, "| available:", torch.cuda.is_available())
report = separator.device_report(refresh=True)
print("device_report:", report["device"], "| autocast:", report["autocast"])
assert report["device"] == "cuda", (
    "Not using the GPU — pip probably replaced torch with a CPU wheel. "
    "Restart the session and reinstall without upgrading torch."
)
print("OK — inference will run on", torch.cuda.get_device_name(0))


## 4 · Pick an input

Either set `INPUT` to a file (a Kaggle dataset path like
`/kaggle/input/my-song/track.mp3` works fine), or set `YOUTUBE_URL` and let it fetch.
Kaggle's `/kaggle/input` is read-only, so outputs always go to `/kaggle/working`.


In [ ]:
import pathlib

INPUT = ""          # e.g. "/kaggle/input/my-song/track.mp3"
YOUTUBE_URL = ""    # e.g. "https://www.youtube.com/watch?v=..."

src = None
if INPUT and pathlib.Path(INPUT).exists():
    src = pathlib.Path(INPUT)
elif YOUTUBE_URL:
    from stemkit import youtube
    out = pathlib.Path("/kaggle/working/input"); out.mkdir(exist_ok=True)
    src = pathlib.Path(youtube.download(YOUTUBE_URL, out)["path"])
else:
    # fall back to any audio file attached as a dataset
    for pat in ("*.mp3", "*.wav", "*.flac", "*.m4a"):
        hits = sorted(pathlib.Path("/kaggle/input").rglob(pat))
        if hits:
            src = hits[0]
            break

assert src, "Set INPUT or YOUTUBE_URL above, or attach an audio file as a Kaggle dataset."
print("input:", src, "|", round(src.stat().st_size / 1048576, 1), "MB")


## 5 · Separate (engine API) — the fast path

In [ ]:
import time
from IPython.display import Audio, display
from stemkit import separator

t0 = time.monotonic()
result = separator.separate_file(
    str(src),
    "2stem",                      # "2stem" = Vocals + Instrumental · "6stem" = Vocals/Drums/Bass/Guitar/Piano/Other
    "/kaggle/working/stems",
    quality="studio",             # "studio" = best quality · "fast" = quicker
)
print(f"{result['mode_name']} · {result['quality_name']} · {result['elapsed']}s · {result['device']}")
print(f"source: {result['duration']}s @ {result['sample_rate']} Hz")
for s in result["stems"]:
    print(f"  {s['name']:14} {s['filename']:26} {s['bytes']/1048576:6.1f} MB  "
          f"{s['sample_rate']} Hz  {s['bit_depth']}-bit  {s['channels']} ch")
    display(Audio(s["path"]))


## 6 · Optional — identical CLI

Handy for batch jobs over a whole album folder:


In [ ]:
%%bash
set -e
cd "$(python -c 'import os;print(os.getcwd())')"
python cli.py --list-modes
# batch example — uncomment and point at your files:
# python cli.py "/kaggle/input/my-song"/*.mp3 --mode 6stem --out /kaggle/working/stems


## 7 · Optional — the studio UI through a tunnel

Kaggle does not expose ports, so bind the app publicly and tunnel it. Keep `STEM_DEVICE` unset
so the CUDA path is chosen automatically.


In [ ]:
RUN_UI = False   # set True to launch the studio UI

if RUN_UI:
    import os, subprocess, time, pathlib
    os.environ.update(STEM_HOST="0.0.0.0", STEM_PORT="8000", STEM_MODEL_DIR=str(repo / "models"),
                      STEM_WORK_DIR=str(repo / "work"))
    log = open("/kaggle/working/app.log", "wb")
    server = subprocess.Popen([sys.executable, "app.py"], stdout=log, stderr=subprocess.STDOUT)
    time.sleep(12)
    print("server up:", server.poll() is None)
    tunnel = subprocess.Popen(["npx", "-y", "localtunnel", "--port", "8000"],
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    time.sleep(10)
    line = tunnel.stdout.readline().strip()
    print("tunnel:", line or "(no URL yet — check npx output)")
    print("app log:", pathlib.Path('/kaggle/working/app.log').read_text()[-800:])


## 8 · Notes for Kaggle

| | |
| --- | --- |
| **GPU time** | 30 h/week quota; Song-length jobs take under a few minutes on a T4 |
| **Session** | 12 h max; the `work/` and `models/` folders live in `/kaggle/working`, so they survive restarts while the session is alive |
| **Disk** | 20 GB — the RoFormer weights are ~610 MB, stems are lossless WAV (~15–40 MB/min) |
| **Internet** | must be On for the model download and for YouTube input |
| **Long tracks** | the local Mac build needs chunking because of an MPS deadlock; on CUDA the full track runs in one pass |
| **Troubleshooting** | full guide in `KAGGLE.md` in the repo |
